# Part 10 · Notebook 02 — Fractional differentiation, microstructure features and a point-in-time store

**Sessions:** S3–S4 (Features I: fractional differentiation & microstructure · Features II & the feature store) · [Lesson plan](../../docs/lessons/PART_10_MACHINE_LEARNING.md) · graded labs in [`labs/part10/`](../../labs/part10/)

**You will:**
1. Compute fractional-differencing weights and find the smallest d that makes a price series stationary.
2. Recover a known bid–ask spread from trade prices with the Roll estimator.
3. Query a point-in-time feature store that handles publication delays and revisions.
4. Catch a look-ahead feature with the truncation test.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with a known structure (a hidden regime, a known autocorrelation, planted importances, pure noise), so you can see what a model can learn, and that it learns nothing from noise.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p10lib.py is in notebooks/part10/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p10lib as p

p.use_course_style()

## 1. Stationary, but with memory

Prices remember everything (non-stationary); returns (`d = 1`) remember nothing. **Fractional differencing** with `0 < d < 1` keeps some memory and can still be stationary. The weights: `w_0 = 1`, `w_k = −w_{k−1}·(d − k + 1)/k`, stopping at the first `|w_k| < thresh` (a fixed-width window). The function returns them **oldest first**, so that `w @ x[t−L+1..t]` is the transformed value at `t`.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def ffd_weights(d, thresh=1e-4):
    w, k = [1.0], 1
    while True:
        w_k = ...                                                    # ✍️
        if abs(w_k) < thresh:
            break
        w.append(w_k)
        k += 1
    return np.array(w[::-1])                                         # oldest first

cases = [(0.4,), (0.9,), (1.0,), (0.3, 1e-3)]
mine = [p.attempt(ffd_weights, *c) for c in cases]
mine = p.check("ffd_weights", mine, [p.ffd_weights(*c) for c in cases])
print(f"d = 0.4: {len(mine[0])} weights, the newest four {np.round(mine[0][-4:], 3)};  d = 1: {mine[2]} (a plain difference)")

`p.min_ffd` tries `d = 0, 0.1, …, 1` on the log price of the regime market and reports the ADF p-value of the transformed series and its correlation with the original:

In [ ]:
mk = p.regime_market()
d_star, table = p.min_ffd(np.log(mk["close"]))
display(table.round(3).T)
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.plot(p.frac_diff(np.log(mk["close"]), d_star), lw=0.8, label=f"d = {d_star}")
ax.plot(np.log(mk["close"]).diff(), lw=0.5, alpha=0.6, label="d = 1 (returns)")
ax.legend(); ax.set_title("fractionally differenced log price vs returns"); plt.show()

The smallest stationary `d` (ADF p < 0.05) is well below 1, and that series still has a correlation of about 0.97 with the price level. Returns keep almost none (0.05): they throw away the memory a model could use.

## 2. The Roll spread

Trades bounce between the bid and the ask, which makes consecutive price changes negatively correlated. Roll (1984): `spread = 2·√(−cov(Δp_t, Δp_{t−1}))` over a rolling window, 0 where the covariance is positive. Use `dp.rolling(window).cov(dp.shift(1))`.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def roll_spread(close, window=20):
    dp = close.diff()
    c = ...                                                          # ✍️ rolling autocovariance of price changes
    return 2 * np.sqrt((-c).clip(lower=0))

rng = np.random.default_rng(1)
mid = pd.Series(50 + np.cumsum(rng.normal(0, 0.05, 3000)))           # the efficient price
trades = mid + 0.10 / 2 * rng.choice([-1, 1], 3000)                  # trades at the bid or the ask: spread 0.10
cases = [(trades, 250), (mid, 250), (trades, 20)]
mine = [p.attempt(roll_spread, *c) for c in cases]
mine = p.check("roll_spread", mine, [p.roll_spread(*c) for c in cases])
print(f"true spread 0.10 → Roll estimate {mine[0].mean():.4f} on trade prices, {mine[1].mean():.4f} on the mid (no bounce)")

The estimator recovers the planted spread from trade prices alone. On the mid price there is no bounce, and what is left is estimation noise.

## 3. What was known, and when

A quarterly EPS number is published about 45 days after the quarter ends, and may be revised later. A backtest that uses it on the quarter-end date has looked into the future. The store keeps every value with `ts` (what it describes) and `available_at` (when it became known). `get(symbols, features, as_of)` must use only rows with `available_at <= as_of`, and among those take the latest `ts`.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
class MyFeatureStore(p.FeatureStore):
    def get(self, symbols, features, as_of):
        known = ...                                                  # ✍️ the rows already available at as_of
        last = known.sort_values("ts").groupby(["symbol", "feature"])["value"].last()
        out = pd.DataFrame(np.nan, index=symbols, columns=features)
        for (s, f), v in last.items():
            if s in out.index and f in out.columns:
                out.loc[s, f] = v
        return out

quarters = pd.to_datetime(["2023-03-31", "2023-06-30", "2023-09-30", "2023-12-31"])
eps = pd.Series([1.00, 1.10, 1.20, 1.30], index=quarters)
stores = []
for cls in (MyFeatureStore, p.FeatureStore):
    fs = cls()
    fs.put("AAA", "eps", eps, delay=pd.Timedelta(days=45))                   # published 45 days after quarter end
    fs.put("AAA", "eps", pd.Series([1.25], index=quarters[2:3]), delay=pd.Timedelta(days=100))   # a later revision
    fs.put("AAA", "price", pd.Series([50.0, 52.0], index=pd.to_datetime(["2023-11-01", "2024-01-02"])))
    stores.append(fs)
dates = ["2023-04-30", "2023-08-01", "2023-11-20", "2024-01-15", "2024-02-20"]
mine = {d: p.attempt(stores[0].get, ["AAA", "BBB"], ["eps", "price"], d) for d in dates}
mine = p.check("FeatureStore.get", mine, {d: stores[1].get(["AAA", "BBB"], ["eps", "price"], d) for d in dates})
pd.DataFrame({d: mine[d].loc["AAA"] for d in dates}).T.rename_axis("as of")

On 1 August the Q2 number (quarter end 30 June) is not out yet, so the store still returns Q1. Now look at 20 November: Q3's 1.20 was published on 14 November, yet the store answers 1.10 (Q2). The revision to 1.25 *replaced* the original row, so the store forgot the 1.20 that everyone could see until the revision arrived in January. This simple store keeps one vintage per date; a real point-in-time store keeps every vintage (a bitemporal table), or backtests silently use numbers that did not exist yet, or lose the ones that did.

## 4. The truncation test

A feature has no look-ahead if its value at `t` is the same whether it is computed on the whole history or on the data up to `t` only (`p.truncation_test`). A **centered** rolling mean fails; a trailing mean and fractional differencing pass:

In [ ]:
lp = np.log(mk["close"])
points = list(range(300, 2400, 150))
features = {"trailing 20-day mean": lambda s: s.rolling(20).mean(),
            "centered 20-day mean": lambda s: s.rolling(20, center=True).mean(),
            "frac_diff d = 0.4": lambda s: p.frac_diff(s, 0.4)}
pd.Series({k: len(p.truncation_test(f, lp, points)) for k, f in features.items()}, name=f"failures out of {len(points)}").to_frame()

## Wrap-up

* Fractional differencing: stationarity with most of the memory; choose the smallest `d` that passes ADF.
* Microstructure features (Roll spread, Amihud illiquidity) recover real trading frictions.
* Store every value with the time it became known, and keep every revision; run the truncation test on every feature.
* Graded version: `labs/part10/week33_features` (`frac_diff`, `min_ffd`, `roll_spread`, `amihud`, `FeatureStore`, `truncation_test`).